# EHR-ML reviewer-response experiments (standalone)

Rebuilds only the data matrices (no multi-hour baseline runs), then runs the experiments requested by the reviewers:

* **Step 22** controlled stacking-vs-single comparison: one fixed config for every XGBoost, five Level-2 options on identical Level-1 outputs, optional diverse Level-1 roster, paired folds, Nadeau-Bengio corrected 95% CIs.
* **Step 23** the same comparison at the original study's cohort size (n = 729).
* **Step 24** per-feature coverage distribution and a coverage sweep on matrices that are not pre-filtered at 80%.
* **Step 25** SHAP ranking stability across training folds and leak-free (nested) feature selection.
* **Step 26** leakage-safe LOS landmark tasks (features only from [0, t], patients still in ICU at t).

Run all cells top to bottom. Send back the `step2*.csv` files and the coverage PNGs listed at the end.

In [ ]:
import subprocess, sys

pkgs = [
    'lightgbm', 'xgboost>=1.6', 'scikit-learn>=1.3',
    'pandas', 'numpy', 'matplotlib', 'seaborn', 'scipy', 'shap', 'catboost',
]
for p in pkgs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', p])

print('✅ All packages installed!')


In [ ]:
import os, sys

EHR_ML_DIR = '/kaggle/working/EHR-ML'
if not os.path.exists(EHR_ML_DIR):
    ret = os.system('git clone --quiet https://github.com/ryashpal/EHR-ML.git ' + EHR_ML_DIR)
    print(f'Cloned EHR-ML (return={ret})')
else:
    print('EHR-ML already present')

if EHR_ML_DIR not in sys.path:
    sys.path.insert(0, EHR_ML_DIR)
print(f'sys.path[0] = {sys.path[0]}')


In [ ]:
META_COLS = [
    "subject_id", "hadm_id", "stay_id", "anchor_date",
    "mortality", "los_gt7", "los_gt14", "los_days", "los",
    "cohort", "patientunitstayid", "uniquepid",
    "hospitaldischargestatus", "unitdischargeoffset"
]

# ─── Settings configuration ──────────────────────────────────────────────────
FAST_MODE = False
N_SPLITS = 5
PAPER_FAITHFUL = True
RANDOM_STATE = 42
# ─────────────────────────────────────────────────────────────────────────────

from pathlib import Path









# ─── SET THIS to your uploaded dataset paths ─────────────────────────────────

from pathlib import Path

DATA_ROOT = Path("/kaggle/input/datasets/mansoobezehra")

MIMIC_DIR = DATA_ROOT / "mimic-iv-31/mimic-iv-3.1"
EICU_DIR = DATA_ROOT / "eicu-dir" / "eicu-collaborative-research-database-2.0"

print("MIMIC_DIR:", MIMIC_DIR, "exists:", MIMIC_DIR.exists())
print("EICU_DIR :", EICU_DIR, "exists:", EICU_DIR.exists())
# Auto-fallback: scan all available input datasets
if not MIMIC_DIR.exists():
    print('WARNING: Specified MIMIC_DIR not found. Scanning /kaggle/input/ ...')
    for d in sorted(Path('/kaggle/input').iterdir()):
        if any(x in d.name.lower() for x in ['mimic', 'physio']):
            MIMIC_DIR = d
            print(f'  -> Using MIMIC_DIR: {MIMIC_DIR}')
            break

if not EICU_DIR.exists():
    print('WARNING: Specified EICU_DIR not found. Scanning /kaggle/input/ ...')
    for d in sorted(Path('/kaggle/input').iterdir()):
        if 'eicu' in d.name.lower():
            EICU_DIR = d
            print(f'  -> Using EICU_DIR: {EICU_DIR}')
            break

HOSP_DIR = MIMIC_DIR / 'hosp' if (MIMIC_DIR / 'hosp').exists() else MIMIC_DIR
ICU_DIR  = MIMIC_DIR / 'icu'  if (MIMIC_DIR / 'icu').exists()  else MIMIC_DIR
OUT_DIR  = Path('/kaggle/working/ehr_ml_output')
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'MIMIC_DIR : {MIMIC_DIR}  (exists={MIMIC_DIR.exists()})')
print(f'EICU_DIR  : {EICU_DIR}   (exists={EICU_DIR.exists()})')
print(f'HOSP_DIR  : {HOSP_DIR}   (exists={HOSP_DIR.exists()})')
print(f'ICU_DIR   : {ICU_DIR}    (exists={ICU_DIR.exists()})')
print(f'OUT_DIR   : {OUT_DIR}')

In [ ]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

def load(path, **kw):
    # Load csv or csv.gz, auto-detecting compression.
    p = Path(path)
    for fp in [p, Path(str(p) + '.gz')]:
        if fp.exists():
            kw.setdefault('low_memory', False)
            if str(fp).endswith('.gz'):
                kw['compression'] = 'gzip'
            return pd.read_csv(fp, **kw)
    raise FileNotFoundError(f'Not found: {path}')

print('Loading small MIMIC-IV tables...')
admissions    = load(HOSP_DIR / 'admissions.csv')
patients      = load(HOSP_DIR / 'patients.csv')
diagnoses_icd = load(HOSP_DIR / 'diagnoses_icd.csv')
microbio      = load(HOSP_DIR / 'microbiologyevents.csv',
                     usecols=['subject_id','hadm_id','chartdate','charttime',
                              'spec_type_desc','org_name'])
icustays      = load(ICU_DIR  / 'icustays.csv')

print('Identifying cohort patients and episodes to optimize memory...')
# Temporarily parse and get cohort stay/hadm IDs
admissions_df = admissions.copy()
admissions_df['admittime'] = pd.to_datetime(admissions_df['admittime'])
icustays_df = icustays.copy()
icustays_df['intime']  = pd.to_datetime(icustays_df['intime'])

first_icu = (icustays_df.sort_values('intime')
             .groupby('hadm_id').first().reset_index()
             [['subject_id','hadm_id','stay_id','intime','outtime','los']])

def get_temp_cohort(hadm_df):
    c = (hadm_df
         .merge(admissions_df[['subject_id','hadm_id']], on=['subject_id','hadm_id'], how='inner')
         .merge(first_icu[['subject_id','hadm_id']], on=['subject_id','hadm_id'], how='inner'))
    return c.drop_duplicates(['subject_id','hadm_id'])

# ICD cohort
SEPSIS_ICD9  = ['99591', '99592', '78552']
SEPSIS_ICD10 = ['A419', 'R6520', 'R6521']
icd_hadm   = (diagnoses_icd[diagnoses_icd['icd_code'].isin(SEPSIS_ICD9 + SEPSIS_ICD10)]
              [['subject_id','hadm_id']].drop_duplicates())
icd_temp = get_temp_cohort(icd_hadm)

# Micro cohort (preload without contaminant exclusion to ensure all final patients' data is loaded)
micro_temp_df = microbio.copy()
micro_temp_df = micro_temp_df.dropna(subset=['hadm_id'])
micro_temp_df['hadm_id'] = micro_temp_df['hadm_id'].astype(int)

micro_temp_pos = micro_temp_df[
    micro_temp_df['spec_type_desc'].str.contains('BLOOD', case=False, na=False) &
    micro_temp_df['org_name'].notna() &
    (micro_temp_df['org_name'].str.strip() != '') &
    (micro_temp_df['org_name'].str.upper() != 'CANCELLED')
]
# Merge with first_icu to filter by window (-24 to +72 hours)
micro_temp_pos = micro_temp_pos.merge(first_icu[['hadm_id', 'intime']], on='hadm_id', how='inner')
micro_temp_pos['charttime'] = pd.to_datetime(micro_temp_pos['charttime'])
micro_temp_pos['chartdate'] = pd.to_datetime(micro_temp_pos['chartdate'])
micro_temp_pos['dt_diff_hours'] = (micro_temp_pos['charttime'].fillna(micro_temp_pos['chartdate']) - micro_temp_pos['intime']).dt.total_seconds() / 3600.0
blood_pos = micro_temp_pos[micro_temp_pos['dt_diff_hours'].between(-24, 72)][['subject_id','hadm_id']].drop_duplicates(['subject_id','hadm_id'])
micro_temp = get_temp_cohort(blood_pos)

# Total unique hadm_ids in cohort
cohort_hadms = set(icd_temp['hadm_id']).union(set(micro_temp['hadm_id']))
print(f'  Found {len(cohort_hadms)} total unique hadm_ids in sepsis cohorts')

# Item IDs to filter (vitals and labs)
VITALS_IDS = {220045, 220050, 220051, 220052, 220210, 223761, 220277, 220739, 223900, 223901}
LABS_IDS = {51221, 51222, 51265, 51301, 50912, 50971, 50983, 50893, 50882, 50813, 50868, 51006,
            50931, 50960, 50970, 51237, 51275, 50861, 50878, 50885, 50820, 50818, 50821, 50804}

def load_large_table(path, cols, valid_hadms, valid_itemids, chunksize=1500000):
    p = Path(path)
    filepath = p if p.exists() else Path(str(p) + '.gz')
    if not filepath.exists():
        raise FileNotFoundError(f'Not found: {path}')
    
    print(f'  Reading {filepath.name} in chunks of {chunksize:,}...')
    chunks = []
    for chunk in pd.read_csv(filepath, usecols=cols, chunksize=chunksize, low_memory=False):
        chunk = chunk.dropna(subset=['hadm_id', 'itemid'])
        chunk['hadm_id'] = chunk['hadm_id'].astype(int)
        chunk['itemid'] = chunk['itemid'].astype(int)
        filtered = chunk[chunk['hadm_id'].isin(valid_hadms) & chunk['itemid'].isin(valid_itemids)]
        if not filtered.empty:
            chunks.append(filtered)
            
    if not chunks:
        return pd.DataFrame(columns=cols)
    return pd.concat(chunks, ignore_index=True)

print('Loading large tables with memory-efficient filtering...')
labevents = load_large_table(
    HOSP_DIR / 'labevents.csv',
    cols=['subject_id','hadm_id','itemid','charttime','valuenum'],
    valid_hadms=cohort_hadms,
    valid_itemids=LABS_IDS
)

chartevents = load_large_table(
    ICU_DIR / 'chartevents.csv',
    cols=['subject_id','hadm_id','stay_id','itemid','charttime','valuenum'],
    valid_hadms=cohort_hadms,
    valid_itemids=VITALS_IDS
)

print(f'  admissions    : {admissions.shape}')
print(f'  patients      : {patients.shape}')
print(f'  diagnoses_icd : {diagnoses_icd.shape}')
print(f'  microbiologyevents: {microbio.shape}')
print(f'  labevents     : {labevents.shape}')
print(f'  icustays      : {icustays.shape}')
print(f'  chartevents   : {chartevents.shape}')
print('✅ All tables loaded and filtered!')


In [ ]:
# Parse timestamps
admissions['admittime'] = pd.to_datetime(admissions['admittime'])
admissions['dischtime'] = pd.to_datetime(admissions['dischtime'])
admissions['hospital_expire_flag'] = admissions['hospital_expire_flag'].astype(int)
icustays['intime']  = pd.to_datetime(icustays['intime'])
icustays['outtime'] = pd.to_datetime(icustays['outtime'])

# First ICU stay per admission
first_icu = (icustays.sort_values('intime')
             .groupby('hadm_id').first().reset_index()
             [['subject_id','hadm_id','stay_id','intime','outtime','los']])

def build_cohort(hadm_df, label):
    # Merge admission + first ICU; compute outcome labels.
    c = (hadm_df
         .merge(admissions[['subject_id','hadm_id','admittime','dischtime','hospital_expire_flag']],
                on=['subject_id','hadm_id'], how='inner')
         .merge(first_icu, on=['subject_id','hadm_id'], how='inner'))
    c['mortality']   = c['hospital_expire_flag']
    c['los_gt7']     = (c['los'] > 7).astype(int)
    c['los_gt14']    = (c['los'] > 14).astype(int)
    c['anchor_date'] = c['intime'].dt.date.astype(str)
    c['cohort']      = label
    return c.drop_duplicates(['subject_id','hadm_id'])

# ── ICD cohort ─────────────────────────────────────────────────────────────────
SEPSIS_ICD9  = ['99591', '99592', '78552']
SEPSIS_ICD10 = ['A419', 'R6520', 'R6521']

# Diagnostic logging to print counts at each filter stage
print("Stepwise diagnostic logging for ICD cohort:")
raw_icd = diagnoses_icd[diagnoses_icd['icd_code'].isin(SEPSIS_ICD9 + SEPSIS_ICD10)]
print(f"  Stage 1: Raw ICD match: {raw_icd['subject_id'].nunique():,} patients, {raw_icd['hadm_id'].nunique():,} episodes")

icu_hadms = set(icustays['hadm_id'])
raw_icu = raw_icd[raw_icd['hadm_id'].isin(icu_hadms)]
print(f"  Stage 2: + Has ICU stay: {raw_icu['subject_id'].nunique():,} patients, {raw_icu['hadm_id'].nunique():,} episodes")

raw_adult = raw_icu.merge(patients[['subject_id', 'anchor_age']], on='subject_id', how='inner')
raw_adult = raw_adult[raw_adult['anchor_age'] >= 18]
print(f"  Stage 3: + Adult-only (age >= 18): {raw_adult['subject_id'].nunique():,} patients, {raw_adult['hadm_id'].nunique():,} episodes")

# Variant A: exact sepsis ICD codes + linked ICU stay
icd_matches_A = raw_adult.copy()
eps_A = icd_matches_A['hadm_id'].nunique()
pts_A = icd_matches_A['subject_id'].nunique()

# Variant B: Variant A + seq_num == 1
icd_matches_B = icd_matches_A[icd_matches_A['seq_num'] == 1].copy()
eps_B = icd_matches_B['hadm_id'].nunique()
pts_B = icd_matches_B['subject_id'].nunique()

print(f"\nAudit results:")
print(f"  Variant A (exact codes + ICU stay): {eps_A} episodes, {pts_A} patients")
print(f"  Variant B (Variant A + seq_num == 1): {eps_B} episodes, {pts_B} patients")

target_eps = 729
diff_A = abs(eps_A - target_eps)
diff_B = abs(eps_B - target_eps)

print("\nExtra paper-alignment cohort variants:")

# Variant C: Variant B + first ICU stay per hospital admission
icu_first_hadm = (
    icustays.sort_values(["hadm_id", "intime"])
    .drop_duplicates("hadm_id", keep="first")
)
variant_b_df = icd_matches_B.merge(
    icu_first_hadm[["hadm_id", "stay_id", "intime"]],
    on="hadm_id", how="inner"
)
icd_matches_C = icd_matches_B[
    icd_matches_B["hadm_id"].isin(icu_first_hadm["hadm_id"])
].copy()
eps_C = icd_matches_C["hadm_id"].nunique()
pts_C = icd_matches_C["subject_id"].nunique()
print(f"  Variant C = B + first ICU stay per admission: {eps_C} episodes, {pts_C} patients")

# Variant D: Variant C + ICU admission within 48h of hospital admission
icd_matches_C_merged = icd_matches_C.merge(
    admissions[["hadm_id", "admittime"]], on="hadm_id", how="inner"
).merge(
    first_icu[["hadm_id", "intime"]], on="hadm_id", how="inner"
)
icd_matches_C_merged["hours_admit_to_icu"] = (
    pd.to_datetime(icd_matches_C_merged["intime"]) -
    pd.to_datetime(icd_matches_C_merged["admittime"])
).dt.total_seconds() / 3600
icd_matches_D_hadms = icd_matches_C_merged[
    icd_matches_C_merged["hours_admit_to_icu"].between(0, 48)
]["hadm_id"].unique()
icd_matches_D = icd_matches_C[icd_matches_C["hadm_id"].isin(icd_matches_D_hadms)].copy()
eps_D = icd_matches_D["hadm_id"].nunique()
pts_D = icd_matches_D["subject_id"].nunique()
print(f"  Variant D = C + ICU within 48h of hospital admission: {eps_D} episodes, {pts_D} patients")

# Variant E: Variant D + first sepsis admission per patient
icd_matches_D_merged = icd_matches_D.merge(
    admissions[["hadm_id", "admittime"]], on="hadm_id", how="left"
)
icd_matches_E = (
    icd_matches_D_merged.sort_values(["subject_id", "admittime"])
    .drop_duplicates("subject_id", keep="first")
)
eps_E = icd_matches_E["hadm_id"].nunique()
pts_E = icd_matches_E["subject_id"].nunique()
print(f"  Variant E = D + first sepsis admission per patient: {eps_E} episodes, {pts_E} patients")

# Choose variant closest to paper target (691 patients / 729 episodes)
variants = {
    'A': (eps_A, pts_A, icd_matches_A),
    'B': (eps_B, pts_B, icd_matches_B),
    'C': (eps_C, pts_C, icd_matches_C),
    'D': (eps_D, pts_D, icd_matches_D),
    'E': (eps_E, pts_E, icd_matches_E),
}
best_var = min(variants, key=lambda k: abs(variants[k][0] - target_eps))
eps_best, pts_best, chosen_raw = variants[best_var]
print(f"\nSelecting Variant {best_var} (closest to {target_eps} episodes): {eps_best} eps, {pts_best} pts")

icd_hadm = chosen_raw[['subject_id', 'hadm_id']].drop_duplicates()
icd_cohort = build_cohort(icd_hadm, 'ICD')

print('\nMIMIC-IV ICD Cohort:')
print(f'  Patients : {icd_cohort["subject_id"].nunique():,}  (paper: 691)')
print(f'  Episodes : {icd_cohort.shape[0]:,}  (paper: 729)')
print(f'  Mortality: {icd_cohort["mortality"].mean():.1%}')
print(f'  LOS>7    : {icd_cohort["los_gt7"].mean():.1%}')
print(f'  LOS>14   : {icd_cohort["los_gt14"].mean():.1%}')


In [ ]:
# ── Micro cohort ───────────────────────────────────────────────────────────────
print("Stepwise diagnostic logging for Micro cohort:")
raw_micro = microbio.dropna(subset=['hadm_id'])
raw_micro['hadm_id'] = raw_micro['hadm_id'].astype(int)
print(f"  Stage 1: Raw microbio events: {raw_micro['subject_id'].nunique():,} patients, {raw_micro['hadm_id'].nunique():,} episodes")

blood_micro = raw_micro[raw_micro['spec_type_desc'].str.contains('BLOOD', case=False, na=False)]
print(f"  Stage 2: + Blood cultures: {blood_micro['subject_id'].nunique():,} patients, {blood_micro['hadm_id'].nunique():,} episodes")

pos_micro = blood_micro[blood_micro['org_name'].notna() & (blood_micro['org_name'].str.strip() != '') & (blood_micro['org_name'].str.upper() != 'CANCELLED')]
print(f"  Stage 3: + Positive blood cultures: {pos_micro['subject_id'].nunique():,} patients, {pos_micro['hadm_id'].nunique():,} episodes")

contaminants = [
    'STAPHYLOCOCCUS, COAGULASE NEGATIVE', 'COAGULASE NEGATIVE STAPHYLOCOCCUS',
    'STAPHYLOCOCCUS EPIDERMIDIS', 'STAPHYLOCOCCUS HOMINIS', 'STAPHYLOCOCCUS CAPITIS',
    'STAPHYLOCOCCUS WARNERI', 'STAPHYLOCOCCUS SAPROPHYTICUS', 'STAPHYLOCOCCUS COHNII',
    'STAPHYLOCOCCUS SIMULANS', 'STAPHYLOCOCCUS XYLOSUS', 'STAPHYLOCOCCUS HAEMOLYTICUS',
    'STAPHYLOCOCCUS LUGDUNENSIS', 'CORYNEBACTERIUM SPECIES (DIPHTHEROIDS)',
    'CORYNEBACTERIUM SPECIES', 'CORYNEBACTERIUM', 'PROPIONIBACTERIUM ACNES',
    'CUTIBACTERIUM ACNES', 'MICROCOCCUS', 'BACILLUS SPECIES', 'BACILLUS'
]
clean_micro = pos_micro[~pos_micro['org_name'].str.upper().isin(contaminants)]
print(f"  Stage 4a: Positive blood cultures (no contaminant exclusion): {pos_micro['subject_id'].nunique():,} patients, {pos_micro['hadm_id'].nunique():,} episodes")
print(f"  Stage 4b: Positive blood cultures (with contaminant exclusion): {clean_micro['subject_id'].nunique():,} patients, {clean_micro['hadm_id'].nunique():,} episodes")

# Final cohort uses pos_micro (all positive blood cultures, no contaminant exclusion) to align with paper
blood_pos = pos_micro.merge(first_icu[['hadm_id', 'intime']], on='hadm_id', how='inner')
blood_pos['charttime'] = pd.to_datetime(blood_pos['charttime'])
blood_pos['chartdate'] = pd.to_datetime(blood_pos['chartdate'])
blood_pos['dt_diff_hours'] = (blood_pos['charttime'].fillna(blood_pos['chartdate']) - blood_pos['intime']).dt.total_seconds() / 3600.0
blood_pos = blood_pos[blood_pos['dt_diff_hours'].between(-24, 72)]
blood_pos = blood_pos[['subject_id','hadm_id','chartdate']].drop_duplicates(['subject_id','hadm_id'])

micro_cohort = build_cohort(blood_pos[['subject_id','hadm_id']], 'Micro')

micro_cohort = micro_cohort.merge(
    blood_pos[['subject_id','hadm_id','chartdate']],
    on=['subject_id','hadm_id'], how='left')
micro_cohort['anchor_date'] = micro_cohort['chartdate'].fillna(
    micro_cohort['intime']).dt.date.astype(str)

print('MIMIC-IV Micro Cohort:')
print(f'  Patients : {micro_cohort["subject_id"].nunique():,}  (paper: 1790)')
print(f'  Episodes : {micro_cohort.shape[0]:,}  (paper: 1790)')
print(f'  Mortality: {micro_cohort["mortality"].mean():.1%}')


In [ ]:
VITALS = {
    220045:'heart_rate', 220050:'sbp', 220051:'dbp', 220052:'mbp',
    220210:'resp_rate', 223761:'temperature_f', 220277:'spo2',
    220739:'gcs_eye', 223900:'gcs_verbal', 223901:'gcs_motor',
}
LABS = {
    51221:'hematocrit', 51222:'hemoglobin', 51265:'platelet', 51301:'wbc',
    50912:'creatinine', 50971:'potassium', 50983:'sodium', 50893:'calcium',
    50882:'bicarbonate', 50813:'lactate', 50868:'anion_gap', 51006:'bun',
    50931:'glucose', 50960:'magnesium', 50970:'phosphate', 51237:'inr',
    51275:'ptt', 50861:'alt', 50878:'ast', 50885:'bilirubin_total',
    50820:'ph', 50818:'pco2', 50821:'po2', 50804:'tco2',
}
# SAPS-II variables (for Table 2 benchmarking)
SAPS2_VITALS = ['heart_rate','sbp','resp_rate','temperature_f','gcs_eye','gcs_verbal','gcs_motor']
SAPS2_LABS   = ['bun','sodium','potassium','bicarbonate','bilirubin_total','wbc','po2']

print(f'Vital sign features : {len(VITALS)}')
print(f'Lab result features : {len(LABS)}')


In [ ]:
from pathlib import Path

def find_real_file(root, name):
    hits = [p for p in Path(root).rglob(name) if p.is_file()]
    if not hits:
        raise FileNotFoundError(f"{name} not found under {root}")
    return hits[0]

eicu_diag_path  = find_real_file(EICU_DIR, "diagnosis.csv")
eicu_pat_path   = find_real_file(EICU_DIR, "patient.csv")
eicu_vital_path = find_real_file(EICU_DIR, "vitalPeriodic.csv")
eicu_lab_path   = find_real_file(EICU_DIR, "lab.csv")

print("eICU diagnosis:", eicu_diag_path)
print("eICU patient  :", eicu_pat_path)
print("eICU vitals   :", eicu_vital_path)
print("eICU lab      :", eicu_lab_path)

In [ ]:
def agg_wide(df, id_cols, feat_col, val_col, prefix, time_col='charttime'):
    # Long -> wide with 5 aggregations: avg, min, max, first, last.
    df = df.copy()
    parts = []
    df_s = df.sort_values(id_cols + [time_col])
    for agg_name, fn in [('avg','mean'),('min','min'),('max','max'),
                          ('first','first'),('last','last')]:
        if agg_name in ('first','last'):
            g = df_s.groupby(id_cols + [feat_col])[val_col].agg(fn).reset_index()
        else:
            g = df.groupby(id_cols + [feat_col])[val_col].agg(fn).reset_index()
        piv = g.pivot_table(index=id_cols, columns=feat_col, values=val_col).reset_index()
        piv.columns = [c if c in id_cols else f'{prefix}_{c}_{agg_name}' for c in piv.columns]
        parts.append(piv)
    result = parts[0]
    for p in parts[1:]:
        result = result.merge(p, on=id_cols, how='outer')
    return result


def build_dm(cohort_df, cov_thresh=0.80, label='', window_before_days=None, window_after_days=None, anchor='intime'):
    # Build EHR-ML data matrix with coverage filtering and optional windowing.
    print(f'\nBuilding data matrix [{label}] ({cohort_df.shape[0]} episodes, cov≥{cov_thresh:.0%})...')
    if window_before_days is not None or window_after_days is not None:
        wb_str = f"WB={window_before_days}" if window_before_days is not None else "WB=None"
        wa_str = f"WA={window_after_days}" if window_after_days is not None else "WA=None"
        print(f'  Applying window filter: {wb_str}, {wa_str} relative to anchor={anchor}')
    
    ids = cohort_df['hadm_id'].unique()
    id_cols = ['subject_id','hadm_id']

    # Merge with cohort_df to get anchor time for filtering
    cohort_times = cohort_df[['hadm_id', anchor]].drop_duplicates().copy()
    cohort_times['anchor_time'] = pd.to_datetime(cohort_times[anchor])

    ce = chartevents[chartevents['hadm_id'].isin(ids) & chartevents['itemid'].isin(VITALS)].copy()
    ce['feature']   = ce['itemid'].map(VITALS)
    ce['charttime'] = pd.to_datetime(ce['charttime'])
    ce = ce[ce['valuenum'].notna() & (ce['valuenum'] > 0)]
    
    if window_before_days is not None or window_after_days is not None:
        ce = ce.merge(cohort_times[['hadm_id', 'anchor_time']], on='hadm_id', how='inner')
        ce['dfa'] = (ce['charttime'] - ce['anchor_time']).dt.total_seconds() / 86400.0
        if window_before_days is not None:
            ce = ce[ce['dfa'] >= -window_before_days]
        if window_after_days is not None:
            ce = ce[ce['dfa'] <= window_after_days]

    vw = agg_wide(ce, id_cols, 'feature', 'valuenum', 'vitals') if len(ce) else pd.DataFrame()

    le = labevents[labevents['hadm_id'].isin(ids) & labevents['itemid'].isin(LABS)].copy()
    le['feature']   = le['itemid'].map(LABS)
    le['charttime'] = pd.to_datetime(le['charttime'])
    le = le[le['valuenum'].notna() & (le['valuenum'] >= 0)]

    if window_before_days is not None or window_after_days is not None:
        le = le.merge(cohort_times[['hadm_id', 'anchor_time']], on='hadm_id', how='inner')
        le['dfa'] = (le['charttime'] - le['anchor_time']).dt.total_seconds() / 86400.0
        if window_before_days is not None:
            le = le[le['dfa'] >= -window_before_days]
        if window_after_days is not None:
            le = le[le['dfa'] <= window_after_days]

    lw = agg_wide(le, id_cols, 'feature', 'valuenum', 'labs') if len(le) else pd.DataFrame()

    if not vw.empty and not lw.empty:
        feats = vw.merge(lw, on=id_cols, how='outer')
    elif not vw.empty:
        feats = vw
    else:
        feats = lw

    dm = cohort_df[[c for c in cohort_df.columns if c in META_COLS]].merge(feats, on=id_cols, how='left')

    feat_cols = [c for c in dm.columns if c not in META_COLS]
    coverage  = dm[feat_cols].notna().mean()
    keep      = coverage[coverage >= cov_thresh].index.tolist()

    print(f'  Total features        : {len(feat_cols)}')
    print(f'  After cov ≥{cov_thresh:.0%}     : {len(keep)}')

    dm = dm[[c for c in META_COLS if c in dm.columns] + keep].dropna(subset=keep, how='all')
    print(f'  Final shape           : {dm.shape}')
    print(f'  Mortality rate        : {dm["mortality"].mean():.1%}')
    return dm

# Build cohort matrices (all admission for Fig 3, 24h window for Fig 4)
dm_icd_all   = build_dm(icd_cohort,   cov_thresh=0.80, label='ICD (all admission)')
dm_icd_24h   = build_dm(icd_cohort,   cov_thresh=0.80, label='ICD (first 24h)', window_before_days=0, window_after_days=1)
dm_icd_24h_hosp = build_dm(icd_cohort, cov_thresh=0.80, label='ICD (first 24h hospital admission)', window_before_days=0, window_after_days=1, anchor='admittime')
dm_micro_all = build_dm(micro_cohort, cov_thresh=0.80, label='Micro (all admission)')
dm_micro_24h = build_dm(micro_cohort, cov_thresh=0.80, label='Micro (first 24h)', window_before_days=0, window_after_days=1)

# Assign default aliases used in downstream code
dm_icd = dm_icd_all
dm_micro = dm_micro_all

dm_icd.to_csv(OUT_DIR / 'data_matrix_icd.csv',   index=False)
dm_micro.to_csv(OUT_DIR / 'data_matrix_micro.csv', index=False)


In [ ]:
import logging
from sklearn.model_selection import train_test_split, cross_validate, GridSearchCV
from sklearn.linear_model   import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics        import (
    roc_auc_score, average_precision_score, matthews_corrcoef,
    f1_score, accuracy_score, balanced_accuracy_score,
    roc_curve, precision_recall_curve, confusion_matrix,
    make_scorer, ConfusionMatrixDisplay
)
from xgboost  import XGBClassifier
from lightgbm import LGBMClassifier
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from matplotlib.patches import Patch

plt.rcParams.update({'figure.dpi':120, 'axes.spines.top':False,
                     'axes.spines.right':False, 'font.size':10})

logging.basicConfig(level=logging.INFO, format='[EHR-ML] %(message)s')
log = logging.getLogger('EHR-ML')

META_COLS = [
    "subject_id", "hadm_id", "stay_id", "anchor_date",
    "mortality", "los_gt7", "los_gt14", "los_days", "los",
    "cohort", "patientunitstayid", "uniquepid",
    "hospitaldischargestatus", "unitdischargeoffset"
]
SCORING = {
    'accuracy':          'accuracy',
    'balanced_accuracy': 'balanced_accuracy',
    'average_precision': 'average_precision',
    'f1':                'f1',
    'roc_auc':           'roc_auc',
    'mcc':               make_scorer(matthews_corrcoef),
}

def preprocess_fold(XTr, XTe, yTr, scale_method=None, remove_outliers=False):
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import StandardScaler, MinMaxScaler
    from sklearn.ensemble import IsolationForest
    import numpy as np

    imp = SimpleImputer(strategy='median')
    XTr = imp.fit_transform(XTr)
    XTe = imp.transform(XTe)

    if remove_outliers:
        iso = IsolationForest(contamination=0.05, random_state=42)
        mask = iso.fit_predict(XTr) == 1
        if len(np.unique(yTr[mask])) > 1:
            XTr, yTr = XTr[mask], yTr[mask]

    if scale_method == 'standard':
        scl = StandardScaler()
        XTr = scl.fit_transform(XTr)
        XTe = scl.transform(XTe)
    elif scale_method == 'minmax':
        scl = MinMaxScaler()
        XTr = scl.fit_transform(XTr)
        XTe = scl.transform(XTe)

    return XTr, XTe, yTr

from sklearn.impute import SimpleImputer
def impute(df, strategy="median"):
    # Impute missing values in a dataframe while preserving index and columns
    imp = SimpleImputer(strategy=strategy)
    arr = imp.fit_transform(df)
    return pd.DataFrame(arr, columns=df.columns, index=df.index)

def get_subsets(dm):
    subsets = {}
    for prefix in ['vitals','labs']:
        for agg in ['avg','min','max','first','last']:
            cols = [c for c in dm.columns if c.startswith(f'{prefix}_') and c.endswith(f'_{agg}')]
            if cols:
                subsets[f'{prefix}_{agg}'] = dm[cols].copy()
    return subsets

def tune_xgb(X, y):
    p = {}
    def best(sp):
        g = GridSearchCV(XGBClassifier(eval_metric='logloss', verbosity=0, **p),
                         sp, cv=3, n_jobs=-1, verbose=0)
        g.fit(X, y)
        return g.cv_results_['params'][list(g.cv_results_['rank_test_score']).index(1)]
    p.update(best({'max_depth': range(1,7), 'scale_pos_weight':[0.1,0.15,0.2,0.25,0.3,0.35,0.4]}))
    p.update(best({'n_estimators': range(50,250,10)}))
    p.update(best({'min_child_weight': range(1,7)}))
    p.update(best({'gamma': [i/10. for i in range(0,5)]}))
    p.update(best({'subsample':[i/10. for i in range(1,10)],
                   'colsample_bytree':[i/10. for i in range(1,10)]}))
    p.update(best({'reg_alpha':[0,1e-5,1e-3,0.1,10]}))
    return p

def tune_lr(X, y):
    g = GridSearchCV(LogisticRegression(max_iter=1000),
                     {'solver':['newton-cg','liblinear'],'C':[100,10,1.0,0.1,0.01]},
                     cv=3, n_jobs=-1)
    g.fit(X, y)
    return g.cv_results_['params'][list(g.cv_results_['rank_test_score']).index(1)]

def standardise_cols(dm, meta_cols):
    aliases = {
        "heart_rate":        "heart_rate",
        "respiratory_rate":  "resp_rate",
        "respiration":       "resp_rate",
        "spo2":              "spo2",
        "sao2":              "spo2",
        "systolic_bp":       "sbp",
        "systemicsystolic":  "sbp",
        "diastolic_bp":      "dbp",
        "systemicdiastolic": "dbp",
        "mean_bp":           "mbp",
        "systemicmean":      "mbp",
        "platelet":          "platelets",
        "platelets":         "platelets",
        "wbc":               "wbc",
        "bun":               "bun",
        "bilirubin_total":   "bilirubin",
        "bilirubin":         "bilirubin",
    }
    rename = {}
    for c in dm.columns:
        if c in meta_cols:
            continue
        new = str(c).lower().replace(" ", "_")
        for old, new_alias in aliases.items():
            new = new.replace(old, new_alias)
        rename[c] = new
    return dm.rename(columns=rename)

print('✅ EHR-ML helpers defined!')


In [ ]:
def qc_clip_features(dm, meta_cols, q_low=0.01, q_high=0.99):
    dm = dm.copy()
    feature_cols = [c for c in dm.columns if c not in meta_cols]
    for c in feature_cols:
        if pd.api.types.is_numeric_dtype(dm[c]):
            lo = dm[c].quantile(q_low)
            hi = dm[c].quantile(q_high)
            dm[c] = dm[c].clip(lo, hi)
    return dm

dm_icd_24h_hosp_qc = qc_clip_features(dm_icd_24h_hosp, META_COLS)
print('MIMIC 24h matrix (QC clipped):', dm_icd_24h_hosp_qc.shape)


In [ ]:
# Reviewer response: build the eICU window matrix at a LOW (50%) coverage threshold and keep the
# pre-filter coverage, so the coverage sweep can see features with 50-80% coverage.
# Only the 72h window is needed here.
COVERAGE_PREFILTER = {}
EICU_BUILD_COV = 0.50

# ============================================================
# Memory-safe eICU ICD cohort + data matrix builder
# Does NOT load full vitalPeriodic.csv into RAM
# ============================================================

from pathlib import Path
import gc
import numpy as np
import pandas as pd

EICU_DIR = Path("/kaggle/input/datasets/mansoobezehra/eicu-dir/eicu-collaborative-research-database-2.0")

def find_real_file(root, name):
    hits = [p for p in Path(root).rglob(name) if p.is_file()]
    if not hits:
        raise FileNotFoundError(f"{name} not found under {root}")
    return hits[0]

patient_path = find_real_file(EICU_DIR, "patient.csv")
diag_path    = find_real_file(EICU_DIR, "diagnosis.csv")
vital_path   = find_real_file(EICU_DIR, "vitalPeriodic.csv")
lab_path     = find_real_file(EICU_DIR, "lab.csv")

print("eICU patient:", patient_path)
print("eICU diagnosis:", diag_path)
print("eICU vitals:", vital_path)
print("eICU labs:", lab_path)

# ----------------------------
# 1. Read only small tables first
# ----------------------------

patient_usecols = [
    "patientunitstayid",
    "uniquepid",
    "age",
    "hospitaldischargestatus",
    "unitdischargeoffset"
]

diag_usecols = [
    "patientunitstayid",
    "icd9code"
]

eicu_patient = pd.read_csv(patient_path, usecols=lambda c: c in patient_usecols)
eicu_diag = pd.read_csv(diag_path, usecols=lambda c: c in diag_usecols)

print("patient:", eicu_patient.shape)
print("diagnosis:", eicu_diag.shape)

# ----------------------------
# 2. Build eICU sepsis ICD cohort
# ----------------------------

def clean_icd_text(x):
    if pd.isna(x):
        return ""
    return str(x).replace(".", "").lower()

eicu_diag["icd_clean"] = eicu_diag["icd9code"].apply(clean_icd_text)

sepsis_patterns = ["99591", "99592", "78552"]

eicu_sepsis_diag = eicu_diag[
    eicu_diag["icd_clean"].apply(lambda x: any(code in x for code in sepsis_patterns))
].copy()

print("Raw eICU sepsis diagnosis rows:", len(eicu_sepsis_diag))
print("Unique eICU sepsis stays:", eicu_sepsis_diag["patientunitstayid"].nunique())

def parse_age_eicu(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    if x in ["> 89", ">89"]:
        return 90
    try:
        return float(x)
    except:
        return np.nan

eicu_patient["age_num"] = eicu_patient["age"].apply(parse_age_eicu)

eicu_cohort = eicu_patient[
    eicu_patient["patientunitstayid"].isin(eicu_sepsis_diag["patientunitstayid"]) &
    (eicu_patient["age_num"] >= 18)
].copy()

eicu_cohort["mortality"] = (
    eicu_cohort["hospitaldischargestatus"]
    .astype(str)
    .str.lower()
    .str.contains("expired|dead|death", regex=True)
).astype(int)

eicu_cohort["los_days"] = pd.to_numeric(
    eicu_cohort["unitdischargeoffset"], errors="coerce"
) / 1440.0

eicu_cohort["los_gt7"] = (eicu_cohort["los_days"] > 7).astype(int)
eicu_cohort["los_gt14"] = (eicu_cohort["los_days"] > 14).astype(int)

print("\nFinal eICU ICD cohort:")
print("episodes:", len(eicu_cohort))
print("patients :", eicu_cohort["uniquepid"].nunique() if "uniquepid" in eicu_cohort.columns else "NA")
print("mortality:", round(eicu_cohort["mortality"].mean() * 100, 2), "%")
print("Paper eICU target: 9612 patients / 11146 episodes")

stay_ids = set(eicu_cohort["patientunitstayid"].astype(int).tolist())

del eicu_diag, eicu_sepsis_diag
gc.collect()

# ----------------------------
# 3. Chunked aggregation helpers
# ----------------------------

def combine_aggregated_parts(stats_parts, first_parts, last_parts, id_col, feat_col, prefix):
    if not stats_parts:
        return pd.DataFrame({id_col: []})

    stats = pd.concat(stats_parts, ignore_index=True)
    stats = stats.groupby([id_col, feat_col], as_index=False).agg({
        "sum": "sum",
        "count": "sum",
        "min": "min",
        "max": "max"
    })
    stats["avg"] = stats["sum"] / stats["count"]

    first = pd.concat(first_parts, ignore_index=True)
    first = first.sort_values([id_col, feat_col, "time"])
    first = first.groupby([id_col, feat_col], as_index=False).first()

    last = pd.concat(last_parts, ignore_index=True)
    last = last.sort_values([id_col, feat_col, "time"])
    last = last.groupby([id_col, feat_col], as_index=False).last()

    parts = []

    for agg_name in ["avg", "min", "max"]:
        piv = stats.pivot_table(index=id_col, columns=feat_col, values=agg_name).reset_index()
        piv.columns = [c if c == id_col else f"{prefix}_{c}_{agg_name}" for c in piv.columns]
        parts.append(piv)

    piv_first = first.pivot_table(index=id_col, columns=feat_col, values="value").reset_index()
    piv_first.columns = [c if c == id_col else f"{prefix}_{c}_first" for c in piv_first.columns]
    parts.append(piv_first)

    piv_last = last.pivot_table(index=id_col, columns=feat_col, values="value").reset_index()
    piv_last.columns = [c if c == id_col else f"{prefix}_{c}_last" for c in piv_last.columns]
    parts.append(piv_last)

    out = parts[0]
    for p in parts[1:]:
        out = out.merge(p, on=id_col, how="outer")

    return out


# ----------------------------
# 4. Process vitalPeriodic.csv in chunks
# ----------------------------

vital_map = {
    "heartrate": "Heart Rate",
    "respiration": "Respiratory Rate",
    "sao2": "SpO2",
    "temperature": "Temperature",
    "systemicsystolic": "Systolic BP",
    "systemicdiastolic": "Diastolic BP",
    "systemicmean": "Mean BP",
}

vital_header = pd.read_csv(vital_path, nrows=0)
vital_cols_available = set(vital_header.columns)

vital_usecols = ["patientunitstayid", "observationoffset"] + [
    c for c in vital_map.keys() if c in vital_cols_available
]

print("\nVital columns used:", vital_usecols)

vital_stats_parts = []
vital_first_parts = []
vital_last_parts = []

chunk_size = 250_000

for i, chunk in enumerate(pd.read_csv(vital_path, usecols=vital_usecols, chunksize=chunk_size)):
    chunk = chunk[chunk["patientunitstayid"].isin(stay_ids)]
    if chunk.empty:
        continue

    long = chunk.melt(
        id_vars=["patientunitstayid", "observationoffset"],
        value_vars=[c for c in vital_usecols if c not in ["patientunitstayid", "observationoffset"]],
        var_name="raw_feature",
        value_name="value"
    )

    long["feature"] = long["raw_feature"].map(vital_map)
    long["value"] = pd.to_numeric(long["value"], errors="coerce")
    long = long.dropna(subset=["feature", "value", "observationoffset"])
    long["value"] = long["value"].astype("float32")

    if long.empty:
        continue

    g = long.groupby(["patientunitstayid", "feature"])["value"].agg(
        sum="sum",
        count="count",
        min="min",
        max="max"
    ).reset_index()
    vital_stats_parts.append(g)

    first_cand = (
        long.sort_values(["patientunitstayid", "feature", "observationoffset"])
        .groupby(["patientunitstayid", "feature"], as_index=False)
        .first()[["patientunitstayid", "feature", "observationoffset", "value"]]
        .rename(columns={"observationoffset": "time"})
    )
    last_cand = (
        long.sort_values(["patientunitstayid", "feature", "observationoffset"])
        .groupby(["patientunitstayid", "feature"], as_index=False)
        .last()[["patientunitstayid", "feature", "observationoffset", "value"]]
        .rename(columns={"observationoffset": "time"})
    )

    vital_first_parts.append(first_cand)
    vital_last_parts.append(last_cand)

    if i % 20 == 0:
        print(f"Processed vital chunk {i}")

    del chunk, long, g, first_cand, last_cand
    gc.collect()

eicu_wide_v = combine_aggregated_parts(
    vital_stats_parts,
    vital_first_parts,
    vital_last_parts,
    id_col="patientunitstayid",
    feat_col="feature",
    prefix="vitals"
)

print("eICU wide vitals:", eicu_wide_v.shape)

del vital_stats_parts, vital_first_parts, vital_last_parts
gc.collect()

# ----------------------------
# 5. Process lab.csv in chunks
# ----------------------------

def map_lab_name(x):
    x = str(x).lower()
    if "wbc" in x:
        return "WBC"
    if "hemoglobin" in x or x == "hgb":
        return "Hemoglobin"
    if "platelet" in x:
        return "Platelets"
    if "sodium" in x:
        return "Sodium"
    if "potassium" in x:
        return "Potassium"
    if "creatinine" in x:
        return "Creatinine"
    if x == "bun" or "blood urea" in x:
        return "BUN"
    if "glucose" in x:
        return "Glucose"
    if "bilirubin" in x:
        return "Bilirubin"
    if "lactate" in x:
        return "Lactate"
    return None

lab_usecols = ["patientunitstayid", "labresultoffset", "labname", "labresult"]

lab_stats_parts = []
lab_first_parts = []
lab_last_parts = []

for i, chunk in enumerate(pd.read_csv(lab_path, usecols=lab_usecols, chunksize=250_000)):
    chunk = chunk[chunk["patientunitstayid"].isin(stay_ids)]
    if chunk.empty:
        continue

    chunk["feature"] = chunk["labname"].apply(map_lab_name)
    chunk["value"] = pd.to_numeric(chunk["labresult"], errors="coerce")
    chunk = chunk.dropna(subset=["feature", "value", "labresultoffset"])
    chunk["value"] = chunk["value"].astype("float32")

    if chunk.empty:
        continue

    g = chunk.groupby(["patientunitstayid", "feature"])["value"].agg(
        sum="sum",
        count="count",
        min="min",
        max="max"
    ).reset_index()
    lab_stats_parts.append(g)

    first_cand = (
        chunk.sort_values(["patientunitstayid", "feature", "labresultoffset"])
        .groupby(["patientunitstayid", "feature"], as_index=False)
        .first()[["patientunitstayid", "feature", "labresultoffset", "value"]]
        .rename(columns={"labresultoffset": "time"})
    )
    last_cand = (
        chunk.sort_values(["patientunitstayid", "feature", "labresultoffset"])
        .groupby(["patientunitstayid", "feature"], as_index=False)
        .last()[["patientunitstayid", "feature", "labresultoffset", "value"]]
        .rename(columns={"labresultoffset": "time"})
    )

    lab_first_parts.append(first_cand)
    lab_last_parts.append(last_cand)

    if i % 10 == 0:
        print(f"Processed lab chunk {i}")

    del chunk, g, first_cand, last_cand
    gc.collect()

eicu_wide_l = combine_aggregated_parts(
    lab_stats_parts,
    lab_first_parts,
    lab_last_parts,
    id_col="patientunitstayid",
    feat_col="feature",
    prefix="labs"
)

print("eICU wide labs:", eicu_wide_l.shape)

del lab_stats_parts, lab_first_parts, lab_last_parts
gc.collect()

# ----------------------------
# 6. Final eICU data matrix
# ----------------------------

dm_eicu = eicu_cohort[
    ["patientunitstayid", "uniquepid", "mortality", "los_gt7", "los_gt14"]
].copy()

dm_eicu["anchor_date"] = "2000-01-01"
dm_eicu["cohort"] = "eICU"

dm_eicu = dm_eicu.merge(eicu_wide_v, on="patientunitstayid", how="left")
dm_eicu = dm_eicu.merge(eicu_wide_l, on="patientunitstayid", how="left")

dm_eicu = dm_eicu.rename(columns={
    "patientunitstayid": "hadm_id",
    "uniquepid": "subject_id"
})

META_COLS_LOCAL = [
    "subject_id", "hadm_id", "anchor_date",
    "mortality", "los_gt7", "los_gt14", "cohort"
]

feature_cols = [c for c in dm_eicu.columns if c not in META_COLS_LOCAL]

coverage = dm_eicu[feature_cols].notna().mean()
keep_cols = coverage[coverage >= 0.80].index.tolist()

dm_eicu = dm_eicu[META_COLS_LOCAL + keep_cols].dropna(subset=keep_cols, how="all")

print("\neICU data matrix:", dm_eicu.shape)
print("eICU feature cols kept:", len(keep_cols))
print("eICU mortality:", round(dm_eicu["mortality"].mean() * 100, 2), "%")
print("Kept features:", keep_cols)

dm_eicu.to_csv(OUT_DIR / "data_matrix_eicu.csv", index=False)
print("Saved:", OUT_DIR / "data_matrix_eicu.csv")

# ── Build 24h-windowed eICU matrix for Figure 5 (paper-comparable) ───────────
# ── Build windowed eICU matrix for Figure 5 sensitivity ───────────────

def build_eicu_window_matrix(window_hours):
    print(f"\nBuilding {window_hours}h-windowed eICU matrix...")
    window_mins = window_hours * 60

    # ----------------------------
    # Windowed vitals
    # ----------------------------
    vital_stats, vital_first, vital_last = [], [], []

    for i, chunk in enumerate(pd.read_csv(vital_path, usecols=vital_usecols, chunksize=250_000)):
        chunk = chunk[
            chunk["patientunitstayid"].isin(stay_ids) &
            (chunk["observationoffset"] >= 0) &
            (chunk["observationoffset"] <= window_mins)
        ]

        if chunk.empty:
            continue

        long = chunk.melt(
            id_vars=["patientunitstayid", "observationoffset"],
            value_vars=[c for c in vital_usecols if c not in ["patientunitstayid", "observationoffset"]],
            var_name="raw_feature",
            value_name="value"
        )

        long["feature"] = long["raw_feature"].map(vital_map)
        long["value"] = pd.to_numeric(long["value"], errors="coerce")
        long = long.dropna(subset=["feature", "value", "observationoffset"])
        long["value"] = long["value"].astype("float32")

        if long.empty:
            continue

        g = long.groupby(["patientunitstayid", "feature"])["value"].agg(
            sum="sum",
            count="count",
            min="min",
            max="max"
        ).reset_index()

        vital_stats.append(g)

        vital_first.append(
            long.sort_values(["patientunitstayid", "feature", "observationoffset"])
            .groupby(["patientunitstayid", "feature"], as_index=False)
            .first()[["patientunitstayid", "feature", "observationoffset", "value"]]
            .rename(columns={"observationoffset": "time"})
        )

        vital_last.append(
            long.sort_values(["patientunitstayid", "feature", "observationoffset"])
            .groupby(["patientunitstayid", "feature"], as_index=False)
            .last()[["patientunitstayid", "feature", "observationoffset", "value"]]
            .rename(columns={"observationoffset": "time"})
        )

        if i % 20 == 0:
            print(f"Processed {window_hours}h vital chunk {i}")

        del chunk, long, g
        gc.collect()

    eicu_wide_v_window = combine_aggregated_parts(
        vital_stats,
        vital_first,
        vital_last,
        id_col="patientunitstayid",
        feat_col="feature",
        prefix="vitals"
    )

    del vital_stats, vital_first, vital_last
    gc.collect()

    print(f"eICU {window_hours}h vitals shape:", eicu_wide_v_window.shape)

    # ----------------------------
    # Windowed labs
    # ----------------------------
    lab_stats, lab_first, lab_last = [], [], []

    lab_usecols_local = ["patientunitstayid", "labresultoffset", "labname", "labresult"]

    for i, chunk in enumerate(pd.read_csv(lab_path, usecols=lab_usecols_local, chunksize=250_000)):
        chunk = chunk[
            chunk["patientunitstayid"].isin(stay_ids) &
            (chunk["labresultoffset"] >= 0) &
            (chunk["labresultoffset"] <= window_mins)
        ]

        if chunk.empty:
            continue

        chunk["feature"] = chunk["labname"].apply(map_lab_name)
        chunk["value"] = pd.to_numeric(chunk["labresult"], errors="coerce")
        chunk = chunk.dropna(subset=["feature", "value", "labresultoffset"])
        chunk["value"] = chunk["value"].astype("float32")

        if chunk.empty:
            continue

        g = chunk.groupby(["patientunitstayid", "feature"])["value"].agg(
            sum="sum",
            count="count",
            min="min",
            max="max"
        ).reset_index()

        lab_stats.append(g)

        lab_first.append(
            chunk.sort_values(["patientunitstayid", "feature", "labresultoffset"])
            .groupby(["patientunitstayid", "feature"], as_index=False)
            .first()[["patientunitstayid", "feature", "labresultoffset", "value"]]
            .rename(columns={"labresultoffset": "time"})
        )

        lab_last.append(
            chunk.sort_values(["patientunitstayid", "feature", "labresultoffset"])
            .groupby(["patientunitstayid", "feature"], as_index=False)
            .last()[["patientunitstayid", "feature", "labresultoffset", "value"]]
            .rename(columns={"labresultoffset": "time"})
        )

        if i % 10 == 0:
            print(f"Processed {window_hours}h lab chunk {i}")

        del chunk, g
        gc.collect()

    eicu_wide_l_window = combine_aggregated_parts(
        lab_stats,
        lab_first,
        lab_last,
        id_col="patientunitstayid",
        feat_col="feature",
        prefix="labs"
    )

    del lab_stats, lab_first, lab_last
    gc.collect()

    print(f"eICU {window_hours}h labs shape:", eicu_wide_l_window.shape)

    # ----------------------------
    # Build final windowed matrix
    # ----------------------------
    dm_window = eicu_cohort[
        ["patientunitstayid", "uniquepid", "mortality", "los_gt7", "los_gt14"]
    ].copy()

    dm_window["anchor_date"] = "2000-01-01"
    dm_window["cohort"] = "eICU"

    dm_window = dm_window.merge(eicu_wide_v_window, on="patientunitstayid", how="left")
    dm_window = dm_window.merge(eicu_wide_l_window, on="patientunitstayid", how="left")

    dm_window = dm_window.rename(columns={
        "patientunitstayid": "hadm_id",
        "uniquepid": "subject_id"
    })

    meta_local = [
        "subject_id", "hadm_id", "anchor_date",
        "mortality", "los_gt7", "los_gt14", "cohort"
    ]

    feat_cols = [c for c in dm_window.columns if c not in meta_local]
    coverage = dm_window[feat_cols].notna().mean()
    COVERAGE_PREFILTER[window_hours] = coverage.copy()   # reviewer response: keep pre-filter coverage
    keep_cols = coverage[coverage >= EICU_BUILD_COV].index.tolist()

    dm_window = dm_window[meta_local + keep_cols].dropna(subset=keep_cols, how="all")

    out_path = OUT_DIR / f"data_matrix_eicu_{window_hours}h.csv"
    dm_window.to_csv(out_path, index=False)

    print(f"eICU {window_hours}h matrix:", dm_window.shape)
    print(f"eICU {window_hours}h feature cols:", len(keep_cols))
    print("Saved:", out_path)

    del eicu_wide_v_window, eicu_wide_l_window
    gc.collect()

    return dm_window


# Build multiple eICU windows for Figure 5 sensitivity
dm_eicu_72h = build_eicu_window_matrix(72)



## Step 22: controlled architecture comparison
Definitions only (fast). The experiments run in the driver cell at the end.

In [ ]:
# =============================================================================
# STEP 22: CONTROLLED ARCHITECTURE COMPARISON (reviewer response)
# -----------------------------------------------------------------------------
# Addresses reviewer points:
#   * tuning parity            -> every XGBoost in this step (single model,
#                                 Level-1 member, Level-2 meta-learner) uses ONE
#                                 fixed config; no grid search anywhere.
#   * meta-learner unspecified -> five Level-2 options compared on IDENTICAL
#                                 Level-1 outputs: XGB (library defaults, as in
#                                 the original run), XGB (same fixed config),
#                                 logistic regression, simple mean, best-single.
#   * Level-1 diversity        -> diversity table + an optional 'diverse' roster.
#   * effect sizes / CIs       -> paired deltas with Nadeau-Bengio corrected
#                                 95% CIs (valid for overlapping CV folds).
#   * stacked vs single were scored under different CV protocols before; here
#     every method is scored on the SAME outer folds.
# Requires from earlier cells: get_subsets, META_COLS, RANDOM_STATE.
# =============================================================================
import time
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import roc_auc_score, brier_score_loss
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# One shared, fixed, regularised config for every XGBoost used in this step.
XGB_CFG = dict(n_estimators=150, max_depth=3, learning_rate=0.05,
               subsample=0.8, colsample_bytree=0.8,
               eval_metric='logloss', verbosity=0, n_jobs=-1)


def _feat_cols(dm):
    """Numeric, non-meta columns only (robust to stray string columns)."""
    return [c for c in dm.columns
            if c not in META_COLS and pd.api.types.is_numeric_dtype(dm[c])]


def _xgb(seed):
    return XGBClassifier(random_state=seed, **XGB_CFG)


def _make_l1(algo, seed):
    """Level-1 model factory. Imputation (and scaling where needed) is fitted
    inside each pipeline, i.e. on training data only."""
    imp = SimpleImputer(strategy='median')
    if algo == 'XGB':
        return make_pipeline(imp, _xgb(seed))
    if algo == 'LGBM':
        return make_pipeline(imp, LGBMClassifier(
            n_estimators=150, max_depth=3, learning_rate=0.05, subsample=0.8,
            subsample_freq=1, colsample_bytree=0.8, verbose=-1, n_jobs=-1,
            random_state=seed))
    if algo == 'LR':
        return make_pipeline(imp, StandardScaler(),
                             LogisticRegression(C=1.0, max_iter=1000))
    if algo == 'MLP':
        return make_pipeline(imp, StandardScaler(), MLPClassifier(
            hidden_layer_sizes=(32, 32), max_iter=300, early_stopping=True,
            random_state=seed))
    if algo == 'RF':
        return make_pipeline(imp, RandomForestClassifier(
            n_estimators=200, min_samples_leaf=5, n_jobs=-1, random_state=seed))
    if algo == 'KNN':
        return make_pipeline(imp, StandardScaler(),
                             KNeighborsClassifier(n_neighbors=25))
    if algo == 'GNB':
        return make_pipeline(imp, StandardScaler(), GaussianNB())
    raise ValueError(algo)


ROSTERS = {
    'original': ['XGB', 'LR', 'LGBM', 'MLP'],            # as in the replicated pipeline
    'diverse':  ['XGB', 'LR', 'KNN', 'RF', 'GNB', 'MLP'],  # adds non-boosted families
}


def _ece(y, p, bins=10):
    edges = np.linspace(0, 1, bins + 1)
    idx = np.clip(np.digitize(p, edges[1:-1]), 0, bins - 1)
    e = 0.0
    for b in range(bins):
        m = idx == b
        if m.any():
            e += m.mean() * abs(y[m].mean() - p[m].mean())
    return e


def _logit(p):
    p = np.clip(p, 1e-4, 1 - 1e-4)
    return np.log(p / (1 - p))


def nb_ci(deltas, n_train, n_test, alpha=0.05):
    """Nadeau-Bengio corrected resampled t-interval for the mean of paired
    per-fold differences (accounts for overlapping training sets)."""
    d = np.asarray(deltas, float)
    k = len(d)
    m = d.mean()
    var = d.var(ddof=1)
    se = np.sqrt((1.0 / k + n_test / n_train) * var)
    if se == 0 or not np.isfinite(se):
        return m, m, m, np.nan
    t = stats.t.ppf(1 - alpha / 2, k - 1)
    p = 2 * (1 - stats.t.cdf(abs(m / se), k - 1))
    return m, m - t * se, m + t * se, p


def arch_compare(dm, target_col, roster='original', n_splits=5, n_repeats=3,
                 inner_splits=3, seed=None, verbose=True):
    """Score single models and several Level-2 options on identical folds.

    Returns (summary_df, delta_df, diversity_df, raw_dict).
    delta_df compares every method with 'XGB single (all features)'.
    """
    seed = RANDOM_STATE if seed is None else seed
    feat_cols = _feat_cols(dm)
    X_all = dm[feat_cols].values.astype(float)
    y = dm[target_col].values.astype(int).ravel()
    subsets = {k: v.values.astype(float) for k, v in get_subsets(dm[feat_cols]).items()}
    algos = ROSTERS[roster]
    keys = [(a, s) for s in subsets for a in algos]
    key_names = [f'{a}_{s}' for a, s in keys]

    rskf = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats,
                                   random_state=seed)
    methods = ['XGB single (all features)', 'LR single (all features)',
               'Stack, XGB meta (library defaults)',
               'Stack, XGB meta (fixed config)', 'Stack, LR meta',
               'Mean of Level-1', 'Best single Level-1 (picked in-fold)']
    res = {m: {'auc': [], 'brier': [], 'ece': []} for m in methods}
    div_rows = []
    n_tr_list, n_te_list = [], []
    t0 = time.time()

    for f, (tr, va) in enumerate(rskf.split(X_all, y)):
        ytr, yva = y[tr], y[va]
        n_tr_list.append(len(tr)); n_te_list.append(len(va))
        inner = list(StratifiedKFold(inner_splits, shuffle=True,
                                     random_state=seed + f).split(np.zeros(len(tr)), ytr))
        M_tr = np.zeros((len(tr), len(keys)))
        M_va = np.zeros((len(va), len(keys)))
        for j, (algo, sub) in enumerate(keys):
            Xs = subsets[sub]
            Xtr_s, Xva_s = Xs[tr], Xs[va]
            for itr, iva in inner:
                m = _make_l1(algo, seed).fit(Xtr_s[itr], ytr[itr])
                M_tr[iva, j] = m.predict_proba(Xtr_s[iva])[:, 1]
            m = _make_l1(algo, seed).fit(Xtr_s, ytr)
            M_va[:, j] = m.predict_proba(Xva_s)[:, 1]

        preds = {}
        # single models on all features
        preds['XGB single (all features)'] = _make_l1('XGB', seed).fit(
            X_all[tr], ytr).predict_proba(X_all[va])[:, 1]
        preds['LR single (all features)'] = _make_l1('LR', seed).fit(
            X_all[tr], ytr).predict_proba(X_all[va])[:, 1]
        # Level-2 options on identical Level-1 outputs
        meta_default = XGBClassifier(eval_metric='logloss', verbosity=0,
                                     random_state=seed, n_jobs=-1)  # library defaults
        preds['Stack, XGB meta (library defaults)'] = meta_default.fit(
            M_tr, ytr).predict_proba(M_va)[:, 1]
        preds['Stack, XGB meta (fixed config)'] = _xgb(seed).fit(
            M_tr, ytr).predict_proba(M_va)[:, 1]
        preds['Stack, LR meta'] = LogisticRegression(C=1.0, max_iter=1000).fit(
            _logit(M_tr), ytr).predict_proba(_logit(M_va))[:, 1]
        preds['Mean of Level-1'] = M_va.mean(axis=1)
        oof_auc = [roc_auc_score(ytr, M_tr[:, j]) for j in range(M_tr.shape[1])]
        preds['Best single Level-1 (picked in-fold)'] = M_va[:, int(np.argmax(oof_auc))]

        for mth, p in preds.items():
            res[mth]['auc'].append(roc_auc_score(yva, p))
            res[mth]['brier'].append(brier_score_loss(yva, p))
            res[mth]['ece'].append(_ece(yva, p))

        # diversity of Level-1 outputs on the validation fold
        C = np.corrcoef(M_va.T)
        same_algo, diff_algo = [], []
        for i in range(len(keys)):
            for j in range(i + 1, len(keys)):
                (ai, si), (aj, sj) = keys[i], keys[j]
                if ai == aj:
                    same_algo.append(C[i, j])
                elif si == sj:
                    diff_algo.append(C[i, j])
        iu = np.triu_indices(len(keys), 1)
        div_rows.append({'fold': f, 'mean_corr_all': np.nanmean(C[iu]),
                         'mean_corr_same_algo_diff_subset': np.nanmean(same_algo),
                         'mean_corr_diff_algo_same_subset': np.nanmean(diff_algo)})
        if verbose:
            print(f'  fold {f + 1}/{n_splits * n_repeats} done '
                  f'({time.time() - t0:.0f}s)', flush=True)

    # ---- summaries -----------------------------------------------------------
    n_tr, n_te = float(np.mean(n_tr_list)), float(np.mean(n_te_list))
    rows, drows = [], []
    ref = np.array(res['XGB single (all features)']['auc'])
    for mth in methods:
        a = np.array(res[mth]['auc'])
        rows.append({'method': mth, 'roster': roster,
                     'AUROC': a.mean(), 'AUROC_sd': a.std(ddof=1),
                     'Brier': np.mean(res[mth]['brier']),
                     'ECE': np.mean(res[mth]['ece'])})
        if mth != 'XGB single (all features)':
            m, lo, hi, p = nb_ci(a - ref, n_tr, n_te)
            drows.append({'method': mth, 'roster': roster,
                          'delta_AUROC_vs_XGB_single': m, 'CI95_low': lo,
                          'CI95_high': hi, 'p_corrected_t': p,
                          'n_scores': len(a)})
    summary = pd.DataFrame(rows)
    deltas = pd.DataFrame(drows)
    diversity = pd.DataFrame(div_rows).drop(columns='fold').mean().to_frame('value')
    return summary, deltas, diversity, res


def run_step22(tasks, roster='original', n_repeats=3, out_prefix='step22'):
    """tasks: list of (name, dm_qc, target_col)."""
    all_sum, all_del, all_div = [], [], {}
    for name, dm, tgt in tasks:
        label = name.replace('\n', ' ')
        print('=' * 70); print(f'STEP 22 | {label} | roster={roster}'); print('=' * 70)
        s, d, v, _ = arch_compare(dm, tgt, roster=roster, n_repeats=n_repeats)
        s.insert(0, 'task', label); d.insert(0, 'task', label)
        all_sum.append(s); all_del.append(d); all_div[label] = v['value']
        print(s.round(4).to_string(index=False))
        print(d.round(4).to_string(index=False))
        print(v.round(3).T.to_string())
    S = pd.concat(all_sum); D = pd.concat(all_del); V = pd.DataFrame(all_div).T
    try:
        S.to_csv(OUT_DIR / f'{out_prefix}_{roster}_summary.csv', index=False)
        D.to_csv(OUT_DIR / f'{out_prefix}_{roster}_deltas.csv', index=False)
        V.to_csv(OUT_DIR / f'{out_prefix}_{roster}_diversity.csv')
    except NameError:
        pass
    return S, D, V


## Steps 23-26: helper functions

In [ ]:
# =============================================================================
# STEPS 23-26 (reviewer response). Requires Step 22 cell to have been run
# (uses XGB_CFG, _xgb, nb_ci, arch_compare) plus get_subsets, META_COLS,
# RANDOM_STATE from earlier cells.
# =============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations
from sklearn.model_selection import RepeatedStratifiedKFold, train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score


def stratified_subsample(dm, target_col, n, seed=0):
    if n >= len(dm):
        return dm.copy()
    keep, _ = train_test_split(np.arange(len(dm)), train_size=n,
                               stratify=dm[target_col].values, random_state=seed)
    return dm.iloc[np.sort(keep)].reset_index(drop=True)


def xgb_fold_scores(dm, target_col, cols=None, n_splits=5, n_repeats=3, seed=None):
    """Fixed-config XGBoost, AUROC on every outer fold (same folds for any `cols`,
    because the folds depend only on rows and labels)."""
    seed = RANDOM_STATE if seed is None else seed
    cols = cols if cols is not None else _feat_cols(dm)
    X = dm[cols].values.astype(float)
    y = dm[target_col].values.astype(int).ravel()
    rskf = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=seed)
    out, ntr, nte = [], [], []
    for tr, va in rskf.split(X, y):
        m = make_pipeline(SimpleImputer(strategy='median'), _xgb(seed)).fit(X[tr], y[tr])
        out.append(roc_auc_score(y[va], m.predict_proba(X[va])[:, 1]))
        ntr.append(len(tr)); nte.append(len(va))
    return np.array(out), float(np.mean(ntr)), float(np.mean(nte))


# ------------------------------------------------------------------ STEP 23 --
def size_sensitivity(dm, target_col, n_target=729, n_draws=5, n_repeats=1,
                     roster='original'):
    """Re-run the stack-vs-single comparison on random stratified subsamples of
    the cohort at the ORIGINAL study's size, to test whether the finding depends
    on the replication cohort being ~3.5x larger."""
    rows = []
    for d in range(n_draws):
        sub = stratified_subsample(dm, target_col, n_target, seed=1000 + d)
        print(f'--- size sensitivity draw {d + 1}/{n_draws} (n={len(sub)}) ---')
        s, dl, _, _ = arch_compare(sub, target_col, roster=roster,
                                   n_repeats=n_repeats, verbose=False)
        s = s.set_index('method')
        base = s.loc['XGB single (all features)', 'AUROC']
        for m in s.index:
            rows.append({'draw': d, 'method': m, 'AUROC': s.loc[m, 'AUROC'],
                         'delta_vs_XGB_single': s.loc[m, 'AUROC'] - base,
                         'ECE': s.loc[m, 'ECE']})
    R = pd.DataFrame(rows)
    agg = (R.groupby('method')
             .agg(AUROC_mean=('AUROC', 'mean'), AUROC_sd=('AUROC', 'std'),
                  delta_mean=('delta_vs_XGB_single', 'mean'),
                  delta_sd=('delta_vs_XGB_single', 'std'),
                  draws_below_single=('delta_vs_XGB_single', lambda x: int((x < 0).sum())),
                  ECE_mean=('ECE', 'mean')).reset_index())
    print(agg.round(4).to_string(index=False))
    return R, agg


# ------------------------------------------------------------------ STEP 24 --
def coverage_report(cov_series, name, bands=(0, .5, .6, .7, .8, .9, 1.0001)):
    """Print and plot the per-feature coverage distribution (pre-filter)."""
    cov = pd.Series(cov_series).dropna()
    cnt = pd.cut(cov, bins=list(bands), right=False).value_counts().sort_index()
    print(f'\nCoverage distribution, {name} (n features = {len(cov)})')
    print(cnt.to_string())
    fig, ax = plt.subplots(figsize=(5.2, 3.0))
    ax.hist(cov.values, bins=np.linspace(0, 1, 21), color='#2E8B8B', edgecolor='white')
    for t in (0.7, 0.8, 0.9):
        ax.axvline(t, color='#B0431F', ls='--', lw=1)
    ax.set_xlabel('Fraction of episodes with a recorded value')
    ax.set_ylabel('Number of features')
    ax.set_title(f'Feature coverage, {name}', fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    try:
        plt.savefig(OUT_DIR / f'coverage_hist_{name.replace(" ", "_")}.png', dpi=300)
    except NameError:
        pass
    plt.show()
    return cnt


def coverage_sweep(dm_full, target_col, thresholds=(0.5, 0.6, 0.7, 0.8, 0.9),
                   ref=0.8, n_repeats=3, name=''):
    """dm_full must be the UNFILTERED feature matrix. Columns are filtered by
    coverage computed on it; rows are identical for every threshold, so the CV
    folds are identical and the comparison is paired."""
    feats = _feat_cols(dm_full)
    cov = dm_full[feats].notna().mean()
    scores, ns, ntr, nte = {}, {}, None, None
    for t in thresholds:
        cols = cov[cov >= t].index.tolist()
        if not cols:
            continue
        s, ntr, nte = xgb_fold_scores(dm_full, target_col, cols, n_repeats=n_repeats)
        scores[t], ns[t] = s, len(cols)
    rows = []
    for t, s in scores.items():
        if ref in scores and t != ref:
            m, lo, hi, p = nb_ci(s - scores[ref], ntr, nte)
        else:
            m, lo, hi, p = (0.0, 0.0, 0.0, np.nan)
        rows.append({'dataset': name, 'threshold': t, 'n_features': ns[t],
                     'AUROC': s.mean(), 'delta_vs_ref': m,
                     'CI95_low': lo, 'CI95_high': hi, 'p_corrected_t': p})
    out = pd.DataFrame(rows)
    print(out.round(4).to_string(index=False))
    return out


# ------------------------------------------------------------------ STEP 25 --
def shap_stability_and_nested_selection(dm, target_col, topk=20, ks=(10, 20, 30, 40),
                                        n_splits=5, n_repeats=2, shap_rows=600, seed=None):
    """(a) SHAP feature-ranking stability across training folds.
       (b) Feature selection done INSIDE each training fold (ranking from the
           training fold only), so the selection never sees the validation fold.
       This replaces the earlier ranking that was computed once on all data."""
    import shap
    seed = RANDOM_STATE if seed is None else seed
    feats = _feat_cols(dm)
    X = dm[feats].values.astype(float)
    y = dm[target_col].values.astype(int).ravel()
    rskf = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=seed)
    tops, res = [], {'all': []}
    for k in ks:
        res[k] = []
    ntr, nte = [], []
    rng = np.random.RandomState(seed)
    for f, (tr, va) in enumerate(rskf.split(X, y)):
        ntr.append(len(tr)); nte.append(len(va))
        imp = SimpleImputer(strategy='median').fit(X[tr])
        Xtr, Xva = imp.transform(X[tr]), imp.transform(X[va])
        m = _xgb(seed).fit(Xtr, y[tr])
        samp = rng.choice(len(tr), size=min(shap_rows, len(tr)), replace=False)
        sv = shap.TreeExplainer(m).shap_values(Xtr[samp])
        if isinstance(sv, list):
            sv = sv[-1]
        imp_abs = np.abs(np.asarray(sv)).mean(axis=0)
        order = np.argsort(-imp_abs)
        tops.append(set(order[:topk]))
        res['all'].append(roc_auc_score(y[va], m.predict_proba(Xva)[:, 1]))
        for k in ks:
            sel = order[:min(k, len(order))]
            mk = _xgb(seed).fit(Xtr[:, sel], y[tr])
            res[k].append(roc_auc_score(y[va], mk.predict_proba(Xva[:, sel])[:, 1]))
        print(f'  fold {f + 1}/{n_splits * n_repeats} done', flush=True)
    # stability
    jac = [len(a & b) / len(a | b) for a, b in combinations(tops, 2)]
    freq = pd.Series(0, index=feats, dtype=float)
    for s in tops:
        for i in s:
            freq.iloc[i] += 1
    freq = (freq / len(tops)).sort_values(ascending=False)
    print(f'\nTop-{topk} SHAP set stability across {len(tops)} training folds: '
          f'mean Jaccard = {np.mean(jac):.3f} (min {np.min(jac):.3f})')
    print('Selection frequency (share of folds with the feature in the top set):')
    print(freq.head(topk).round(2).to_string())
    # nested-selection performance
    rows = []
    base = np.array(res['all'])
    for k in ks:
        a = np.array(res[k])
        m_, lo, hi, p = nb_ci(a - base, np.mean(ntr), np.mean(nte))
        rows.append({'top_k': k, 'AUROC': a.mean(), 'AUROC_all_features': base.mean(),
                     'delta_vs_all': m_, 'CI95_low': lo, 'CI95_high': hi,
                     'p_corrected_t': p})
    sel_df = pd.DataFrame(rows)
    print('\nNested (leak-free) top-K selection vs all features:')
    print(sel_df.round(4).to_string(index=False))
    return {'mean_jaccard': float(np.mean(jac)), 'min_jaccard': float(np.min(jac)),
            'freq': freq, 'selection': sel_df}


# ------------------------------------------------------------------ STEP 26 --
def los_landmark_tasks(cohort, landmarks=(1, 2)):
    """Leakage-safe LOS tasks. At landmark day t (ICU admission + t days) we use
    ONLY data from [0, t] and keep ONLY patients still in the ICU at t
    (ICU LOS > t), so the outcome is not already determined and the feature
    window never overlaps the horizon being predicted."""
    tasks = []
    for wa in landmarks:
        dm = build_dm(cohort, cov_thresh=0.80, label=f'ICD LOS landmark {wa}d',
                      window_before_days=0, window_after_days=wa, anchor='intime')
        n0 = len(dm)
        dm = dm[dm['los'] > wa].reset_index(drop=True)
        print(f'  landmark {wa}d: {n0} -> {len(dm)} episodes still in ICU; '
              f'LOS>7 prevalence {dm["los_gt7"].mean():.1%}, '
              f'LOS>14 prevalence {dm["los_gt14"].mean():.1%}')
        dmq = qc_clip_features(dm, META_COLS)
        tasks.append((f'MIMIC LOS>7 (landmark {wa}d)', dmq, 'los_gt7'))
        tasks.append((f'MIMIC LOS>14 (landmark {wa}d)', dmq, 'los_gt14'))
    return tasks


## Driver: run all reviewer-response experiments
Set `QUICK = True` first for a fast pass (1 repeat).

In [ ]:
# =============================================================================
# DRIVER: runs every reviewer-response experiment. Edit the flags, then run.
# =============================================================================
QUICK            = False   # True = 1 repeat instead of 3 (about 3x faster, wider CIs)
STEP22_REPEATS   = 1 if QUICK else 3
EICU_N           = 5000    # stratified eICU subsample for the expensive Level-1 runs
RUN_DIVERSE      = True    # also run the 'diverse' Level-1 roster (MIMIC tasks)
LANDMARKS        = (2,)    # LOS landmark days; (1, 2) runs both

# ---- 0. Restore the original 80% coverage filter on the main eICU matrix ------
def apply_cov_thresh(dm, thresh, meta_cols=META_COLS):
    keep = [c for c in dm.columns if c in meta_cols or dm[c].notna().mean() >= thresh]
    return dm[keep]

dm_eicu_72h_full = dm_eicu_72h.copy()              # built at 50% coverage
dm_eicu_72h      = apply_cov_thresh(dm_eicu_72h_full, 0.80)
n_feat = len([c for c in dm_eicu_72h.columns if c not in META_COLS])
print(f'eICU 72h matrix at 80% coverage: {n_feat} features '
      f'(the original run reported 55)')
dm_eicu_72h_qc = qc_clip_features(dm_eicu_72h, META_COLS)

# ---- 1. Leakage-safe LOS landmark tasks ---------------------------------------
print('\nBuilding LOS landmark tasks ...')
los_tasks = los_landmark_tasks(icd_cohort, landmarks=LANDMARKS)

# ---- 2. STEP 22: controlled architecture comparison ---------------------------
eicu_sub = stratified_subsample(dm_eicu_72h_qc, 'mortality', EICU_N, seed=7)
tasks_main = [
    ('MIMIC-IV 24h-window in-hospital mortality', dm_icd_24h_hosp_qc, 'mortality'),
    (f'eICU 72h-window in-hospital mortality (n={len(eicu_sub)} subsample)', eicu_sub, 'mortality'),
] + [t for t in los_tasks if 'LOS>7' in t[0]]
S_orig, D_orig, V_orig = run_step22(tasks_main, roster='original', n_repeats=STEP22_REPEATS)

if RUN_DIVERSE:
    tasks_div = [tasks_main[0]] + [t for t in los_tasks if 'LOS>7' in t[0]]
    S_div, D_div, V_div = run_step22(tasks_div, roster='diverse', n_repeats=STEP22_REPEATS)

# ---- 3. STEP 23: does the finding depend on cohort size? ----------------------
print('\n' + '=' * 70); print('STEP 23: cohort-size sensitivity (n = 729, original study size)')
R23, A23 = size_sensitivity(dm_icd_24h_hosp_qc, 'mortality', n_target=729,
                            n_draws=5, n_repeats=1)
try:
    R23.to_csv(OUT_DIR / 'step23_size_sensitivity_draws.csv', index=False)
    A23.to_csv(OUT_DIR / 'step23_size_sensitivity_summary.csv', index=False)
except Exception as e:
    print('could not save step 23:', e)

# ---- 4. STEP 24: coverage evidence (replaces the unsupported eICU claim) ------
print('\n' + '=' * 70); print('STEP 24: feature-coverage distribution and sweep')
cov_eicu = COVERAGE_PREFILTER[72]            # coverage BEFORE any filtering
coverage_report(cov_eicu, 'eICU 72h')
dm_mimic24_full = build_dm(icd_cohort, cov_thresh=0.0, label='ICD 24h, unfiltered',
                           window_before_days=0, window_after_days=1, anchor='admittime')
feats_m = [c for c in dm_mimic24_full.columns if c not in META_COLS]
coverage_report(dm_mimic24_full[feats_m].notna().mean(), 'MIMIC-IV 24h')
sw_m = coverage_sweep(dm_mimic24_full, 'mortality', name='MIMIC-IV 24h', n_repeats=STEP22_REPEATS)
sw_e = coverage_sweep(dm_eicu_72h_full, 'mortality', name='eICU 72h', n_repeats=STEP22_REPEATS,
                      thresholds=(0.5, 0.6, 0.7, 0.8, 0.9))
pd.concat([sw_m, sw_e]).to_csv(OUT_DIR / 'step24_coverage_sweep.csv', index=False)

# ---- 5. STEP 25: SHAP stability + leak-free feature selection -----------------
print('\n' + '=' * 70); print('STEP 25: SHAP fold stability and nested feature selection')
shap_res = {}
for nm, dmx in [('MIMIC-IV 24h mortality', dm_icd_24h_hosp_qc),
                ('eICU 72h mortality', eicu_sub)]:
    print(f'\n--- {nm} ---')
    shap_res[nm] = shap_stability_and_nested_selection(dmx, 'mortality',
                                                       n_repeats=2 if not QUICK else 1)
    shap_res[nm]['selection'].to_csv(
        OUT_DIR / f'step25_nested_selection_{nm.split()[0]}.csv', index=False)
    shap_res[nm]['freq'].head(30).to_csv(
        OUT_DIR / f'step25_shap_frequency_{nm.split()[0]}.csv')

# ---- 6. Summary of produced files --------------------------------------------
import glob, os
print('\nFiles to send back for the paper revision:')
for f in sorted(glob.glob(str(OUT_DIR / 'step2[2-5]*')) + glob.glob(str(OUT_DIR / 'coverage_hist*'))):
    print('  ', os.path.basename(f))
